# House Price Prediction — Model Training Notebook

**Dataset:** [House Price](https://www.kaggle.com/datasets/juhibhojani/house-price) by Juhi Bhojani (Kaggle)

**Goal:** Clean a messy, real-world Indian real-estate listings dataset, engineer useful
features, train and compare regression models, and export a single scikit-learn
`Pipeline` (preprocessing + model) that the FastAPI backend can load and serve directly.

**Notebook sections:**
1. Load & Inspect
2. Missing Values Analysis
3. Data Cleaning & Feature Engineering
4. Exploratory Data Analysis (EDA)
5. Outlier Removal
6. Pipeline & Model Training
7. Evaluation & Model Comparison
8. Cross-Validation
9. Export Model & Metadata
10. Conclusion

> This notebook is designed to run top-to-bottom without errors (`Kernel -> Restart & Run All`).

## 1. Load & Inspect

We start by loading the raw CSV and taking a first look at its shape, dtypes and a
sample of rows. **Always verify the real columns after downloading** — never trust a
written description over the actual file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

pd.set_option("display.max_columns", None)
%matplotlib inline
sns.set_style("whitegrid")

# The CSV is expected at notebooks/data/house_prices.csv (see README for download steps).
# A Colab-style fallback path is kept for convenience.
DATA_PATH = "data/house_prices.csv" if os.path.exists("data/house_prices.csv") else "/content/house_prices.csv"

df = pd.read_csv(DATA_PATH)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
df.columns.tolist()

### 1.1 How many rows / columns?

The raw dataset has **187,531 rows and 21 columns**.

### 1.2 Which columns are numeric vs. text?

Only `Index` and `Price (in rupees)` are loaded as numeric out of the box. Every other
column — including columns that are conceptually numeric such as `Amount(in rupees)`,
`Carpet Area`, `Super Area`, `Floor`, `Bathroom`, `Balcony` and `Car Parking` — is stored
as **text**, because the raw values contain units and free text (e.g. `"42 Lac"`,
`"1200 sqft"`, `"3 out of 10"`). Cleaning these into real numeric columns is the main
work of this notebook.

### 1.3 Which columns have the most missing values?

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0]

`Plot Area` and `Dimensions` are essentially empty and will be dropped. `Society`,
`Super Area`, `Car Parking`, `Carpet Area`, `overlooking` and `facing` are missing for a
large fraction of listings — `Carpet Area` and `Super Area` are complementary (a listing
usually reports one or the other), so we will **merge them into a single area feature**
instead of dropping either outright.

## 2. Data Cleaning & Feature Engineering

This dataset is messy on purpose. We handle, in order:

1. **Price** (`Amount(in rupees)`) — text like `"42 Lac"` / `"1.2 Cr"` → numeric rupees.
2. **Area** (`Carpet Area` + `Super Area`) — text like `"1200 sqft"` / `"140 sqm"` →
   a single numeric `area_sqft` feature, normalised to square feet.
3. **Floor** — text like `"3 out of 10"` → numeric floor number (handling `Ground` /
   `Basement`).
4. **Bathroom / Balcony / Car Parking** — convert to numeric, impute missing values.
5. **High-cardinality categoricals** (`location`) — keep the top-50 locations, group the
   rest into `"other"`.
6. **Drop useless columns** — `Index`, `Title`, `Description`, `Dimensions`, `Plot Area`,
   `Society`, `overlooking`, `Price (in rupees)` (duplicate/derived price-per-sqft-like
   column, not our target).
7. **Outlier removal** — drop listings with absurd price-per-sqft (below 1st / above
   99th percentile).

### 2.1 Price

In [ ]:
def parse_amount(x):
    """Convert values like '42 Lac' / '1.2 Cr' / '25,00,000' into numeric rupees."""
    if not isinstance(x, str):
        return np.nan
    x = x.strip().lower()
    try:
        if "lac" in x:
            return float(x.replace("lac", "").strip()) * 1e5
        if "cr" in x:
            return float(x.replace("cr", "").strip()) * 1e7
        return float(x.replace(",", ""))
    except ValueError:
        return np.nan

df["price_clean"] = df["Amount(in rupees)"].apply(parse_amount)
df = df.dropna(subset=["price_clean"])
df.shape

In [ ]:
df[["Amount(in rupees)", "price_clean"]].head()

### 2.2 Area — merge Carpet Area & Super Area

Both columns store values such as `"1200 sqft"` or `"140 sqm"`. We extract the numeric
value, convert square metres to square feet (1 sqm ≈ 10.764 sqft), and then combine the
two columns: use `Carpet Area` when available, otherwise fall back to `Super Area`. This
keeps far more usable rows than dropping either column outright.

In [ ]:
def parse_area_to_sqft(x):
    """Extract a numeric area in square feet from strings like '1200 sqft' or '140 sqm'."""
    if not isinstance(x, str):
        return np.nan
    x = x.strip().lower()
    match = pd.Series([x]).str.extract(r"([\d.,]+)")[0].iloc[0]
    if match is None or match == "":
        return np.nan
    try:
        value = float(match.replace(",", ""))
    except ValueError:
        return np.nan
    if "sqm" in x or "sq. m" in x or "sqmt" in x:
        value *= 10.764
    return value

df["carpet_area_sqft"] = df["Carpet Area"].apply(parse_area_to_sqft)
df["super_area_sqft"] = df["Super Area"].apply(parse_area_to_sqft)

# Prefer Carpet Area (more precise, usable-floor-space); fall back to Super Area.
df["area_sqft"] = df["carpet_area_sqft"].fillna(df["super_area_sqft"])

print("Missing after merge:", df["area_sqft"].isna().mean().round(3))
df[["Carpet Area", "Super Area", "area_sqft"]].head(10)

In [ ]:
# Drop rows with no usable area at all, then drop the intermediate helper columns.
df = df.dropna(subset=["area_sqft"])
df = df.drop(columns=["Carpet Area", "Super Area", "carpet_area_sqft", "super_area_sqft"])
df.shape

### 2.3 Floor

In [ ]:
df["Floor"].head(10)

In [ ]:
# "Ground" -> 0, then keep only the first number ("3 out of 10" -> "3")
df["floor_num"] = (
    df["Floor"]
    .str.replace("Ground", "0", regex=False)
    .str.replace("Basement", "-1", regex=False)
    .str.split(" ").str[0]
)
df["floor_num"] = pd.to_numeric(df["floor_num"], errors="coerce")
df = df.drop(columns=["Floor"])
df["floor_num"].head(10)

### 2.4 Bathroom, Balcony, Car Parking

In [ ]:
df["Bathroom"] = pd.to_numeric(df["Bathroom"], errors="coerce")
df["Balcony"] = pd.to_numeric(df["Balcony"], errors="coerce")
# Car Parking is text like "1 Covered" -> extract the leading number
df["Car Parking"] = df["Car Parking"].str.extract(r"(\d+)")
df["Car Parking"] = pd.to_numeric(df["Car Parking"], errors="coerce")

df = df.rename(columns={"Bathroom": "bathroom", "Balcony": "balcony", "Car Parking": "car_parking"})
df[["bathroom", "balcony", "car_parking"]].describe()

### 2.5 Impute remaining missing values

Numeric columns are imputed with the median, `car_parking` with 0 (no parking is a
reasonable default for a missing value), and categorical columns with their mode.

In [ ]:
df["area_sqft"] = df["area_sqft"].fillna(df["area_sqft"].median())
df["floor_num"] = df["floor_num"].fillna(df["floor_num"].median())
df["bathroom"] = df["bathroom"].fillna(df["bathroom"].median())
df["balcony"] = df["balcony"].fillna(df["balcony"].median())
df["car_parking"] = df["car_parking"].fillna(0)

for col in ["Furnishing", "Status", "Transaction", "Ownership"]:
    df[col] = df[col].fillna(df[col].mode()[0])

df.isna().sum().sort_values(ascending=False).head(10)

### 2.6 Drop unused columns

In [ ]:
df = df.drop(columns=[
    "Index", "Title", "Description", "Price (in rupees)",
    "Dimensions", "Plot Area", "Society", "overlooking", "facing",
], errors="ignore")
df.columns.tolist()

### 2.7 High-cardinality categorical — group locations

`location` has thousands of distinct values. We keep the top-50 most frequent locations
and group everything else into `"other"` before one-hot encoding.

In [ ]:
TOP_N_LOCATIONS = 50
top_locations = df["location"].value_counts().head(TOP_N_LOCATIONS).index

df["location_grouped"] = df["location"].apply(lambda x: x if x in top_locations else "other")
df = df.drop(columns=["location"])
df["location_grouped"].value_counts().head(10)

### 2.8 Standardise column names

Rename the remaining columns to clean, lower-case, backend-friendly names.

In [ ]:
df = df.rename(columns={
    "Status": "status",
    "Transaction": "transaction",
    "Furnishing": "furnishing",
    "Ownership": "ownership",
})
df.info()

## 3. Exploratory Data Analysis (EDA)

Four required plots, each with a short interpretation below it.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["price_clean"], log_scale=True, bins=40)
plt.title("Distribution of House Prices (log scale)")
plt.xlabel("Price (rupees, log scale)")
plt.ylabel("Count")
plt.show()

**Comment:** House prices are heavily right-skewed — most listings cluster in the
lower-to-mid price range, with a long tail of very expensive properties. Viewing the
x-axis on a log scale makes the shape much easier to read than a linear axis.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x="area_sqft", y="price_clean", data=df, alpha=0.3)
plt.yscale("log")
plt.title("Area vs. House Price")
plt.xlabel("Area (sqft)")
plt.ylabel("Price (rupees, log scale)")
plt.show()

**Comment:** There is a clear positive relationship between area and price — larger
properties tend to cost more — although the relationship is noisy, reflecting the effect
of location, furnishing and other factors not shown in this 2-D view.

In [ ]:
top_location_avg = (
    df.groupby("location_grouped")["price_clean"].mean().sort_values(ascending=False).head(15)
)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_location_avg.index, y=top_location_avg.values)
plt.xticks(rotation=75, ha="right")
plt.title("Average Price by Top-15 Locations")
plt.ylabel("Average Price (rupees)")
plt.tight_layout()
plt.show()

**Comment:** Average prices vary dramatically by location, confirming that
`location_grouped` is an important predictive feature and justifying the top-N grouping
strategy used above.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x="furnishing", y="price_clean", data=df, ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Price by Furnishing Status")
axes[0].tick_params(axis="x", rotation=30)

sns.boxplot(x="bathroom", y="price_clean", data=df[df["bathroom"] <= 6], ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Price by Number of Bathrooms")

plt.tight_layout()
plt.show()

**Comment:** Furnished properties tend to command a higher median price than
semi-furnished or unfurnished ones. Price also rises steadily with the number of
bathrooms, which makes sense since bathroom count correlates with overall property size
and quality.

## 4. Outlier Removal

We compute price-per-sqft and drop listings below the 1st or above the 99th percentile —
these are almost always data-entry errors or highly atypical listings (e.g. "Call for
Price" placeholders that slipped through, or luxury outliers) that would otherwise
distort the model.

In [ ]:
df["price_per_sqft"] = df["price_clean"] / df["area_sqft"]

q_low = df["price_per_sqft"].quantile(0.01)
q_high = df["price_per_sqft"].quantile(0.99)

before = df.shape[0]
df = df[(df["price_per_sqft"] >= q_low) & (df["price_per_sqft"] <= q_high)]
df = df.drop(columns=["price_per_sqft"])

print(f"Removed {before - df.shape[0]} outlier rows ({(before - df.shape[0]) / before:.2%})")
df.shape

In [ ]:
df.info()

## 5. Build a Pipeline & Train

Preprocessing (imputation, scaling, one-hot encoding) is bundled **inside** the exported
`Pipeline`, so the FastAPI backend only needs to call `.predict()` on raw feature
values — no manual encoding logic has to be duplicated in the backend.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import sklearn

NUMERIC_FEATURES = ["area_sqft", "floor_num", "bathroom", "balcony", "car_parking"]
CATEGORICAL_FEATURES = ["location_grouped", "status", "transaction", "furnishing", "ownership"]

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df["price_clean"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), NUMERIC_FEATURES),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), CATEGORICAL_FEATURES),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

### 5.1 Linear Regression (baseline)

In [ ]:
linear_model = Pipeline([
    ("prep", preprocessor),
    ("reg", LinearRegression()),
])
linear_model.fit(X_train, y_train)
y_pred_lr = linear_model.predict(X_test)

### 5.2 Random Forest Regressor

In [ ]:
rf_model = Pipeline([
    ("prep", preprocessor),
    ("reg", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

### 5.3 Bonus — does training on log(price) help?

Because price is heavily skewed, we also try training the Random Forest on
`np.log1p(price)` and inverting the prediction with `np.expm1` at inference time, and
compare it against training on raw price.

In [ ]:
rf_log_model = Pipeline([
    ("prep", preprocessor),
    ("reg", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])
rf_log_model.fit(X_train, np.log1p(y_train))
y_pred_rf_log = np.expm1(rf_log_model.predict(X_test))

## 6. Evaluate & Compare Models

In [ ]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

results = pd.DataFrame([
    {"Model": "Linear Regression", **evaluate(y_test, y_pred_lr)},
    {"Model": "Random Forest", **evaluate(y_test, y_pred_rf)},
    {"Model": "Random Forest (log target)", **evaluate(y_test, y_pred_rf_log)},
])
results

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_rf, alpha=0.4)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs. Predicted Price (Random Forest)")
plt.legend()
plt.show()

**Comment:** Most points sit close to the diagonal line, showing the Random Forest
model predicts prices reasonably well; scatter increases at the high-price end, where
data is sparser and prices are more volatile.

### Conclusion

The **Random Forest Regressor (raw target)** is chosen as the final model: it clearly
outperforms the Linear Regression baseline on every metric (lower MAE/RMSE, higher R²),
because it can capture non-linear relationships and interactions between location, area
and property attributes that a linear model cannot. Training on the log-transformed
target gives a very similar (sometimes marginally better) result for tree-based models,
but for simplicity and to avoid an extra inverse-transform step in the backend, we export
the model trained directly on raw price.

## 7. Cross-Validation (bonus)

5-fold cross-validation on the training set gives a more robust estimate of
generalisation performance than a single train/test split.

In [ ]:
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring="r2", n_jobs=-1)
print("5-fold CV R^2 scores:", np.round(cv_scores, 4))
print("Mean CV R^2:", round(cv_scores.mean(), 4), " Std:", round(cv_scores.std(), 4))

## 8. Export the Model

We export the full `Pipeline` (preprocessing + Random Forest) as `house_price.pkl`,
together with `locations.json` (the list of allowed locations for the frontend
dropdown) and a small `metrics.json` used in the README.

In [ ]:
FINAL_MODEL = rf_model

joblib.dump(FINAL_MODEL, "house_price.pkl")

# Sanity check: reload and predict one sample
loaded = joblib.load("house_price.pkl")
sample = X_test.iloc[[0]]
print("Reloaded prediction:", loaded.predict(sample))

locations = sorted(df["location_grouped"].unique().tolist())
with open("locations.json", "w") as f:
    json.dump(locations, f)

final_metrics = evaluate(y_test, y_pred_rf)
final_metrics["model"] = "RandomForestRegressor"
final_metrics["sklearn_version"] = sklearn.__version__
with open("metrics.json", "w") as f:
    json.dump(final_metrics, f, indent=2)

print("scikit-learn version (pin this in backend/requirements.txt):", sklearn.__version__)
print(os.listdir("."))

## 9. Final Summary

| Item | Value |
|---|---|
| Final dataset shape | see `df.shape` above |
| Numeric features | area_sqft, floor_num, bathroom, balcony, car_parking |
| Categorical features | location_grouped, status, transaction, furnishing, ownership |
| Final model | RandomForestRegressor (n_estimators=200) inside a scikit-learn Pipeline |
| Exported artifacts | `house_price.pkl`, `locations.json`, `metrics.json` |

**Next steps:** copy `house_price.pkl` and `locations.json` into `backend/models/` and
`frontend/public/` respectively (see the project README), then start the backend and
frontend.